In [1]:
import torch
from romatch import roma_outdoor  # base architecture
import cv2
import numpy as np


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model with standard weights (we'll overwrite with your weights)
roma = roma_outdoor(device=device)
roma.eval()

2025-11-14 21:16:49.446 | INFO     | romatch.models.model_zoo.roma_models:roma_model:61 - Using coarse resolution (560, 560), and upsample res (864, 864)


RegressionMatcher(
  (encoder): CNNandDinov2(
    (cnn): VGG19(
      (layers): ModuleList(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (9): ReLU(inplace=True)
        (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (12): ReLU(inplace=Tru

In [3]:
ckpt_path = "weights/matchanything_roma.ckpt"  # your file

ckpt = torch.load(ckpt_path, map_location="cpu")

# Common patterns:
#  - ckpt is already a state_dict
#  - or it’s a dict with a "model" or "state_dict" key

if isinstance(ckpt, dict):
    if "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    elif "model" in ckpt:
        state_dict = ckpt["model"]
    else:
        state_dict = ckpt
else:
    state_dict = ckpt

# Optionally strip prefixes like "module." or "model."
def strip_prefix_if_present(state_dict, prefix):
    if not any(k.startswith(prefix) for k in state_dict.keys()):
        return state_dict
    return {k[len(prefix):]: v for k, v in state_dict.items()}

state_dict = strip_prefix_if_present(state_dict, "module.")
state_dict = strip_prefix_if_present(state_dict, "model.")

missing, unexpected = roma.load_state_dict(state_dict, strict=False)
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Missing keys: 541
Unexpected keys: 603


In [4]:
roma.to(device)
roma.eval()

RegressionMatcher(
  (encoder): CNNandDinov2(
    (cnn): VGG19(
      (layers): ModuleList(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (9): ReLU(inplace=True)
        (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (12): ReLU(inplace=Tru

In [5]:
imgA_path = "imgs/rgb0.png"
imgB_path = "imgs/ir1.png"

imgA = cv2.imread(imgA_path)
imgB = cv2.imread(imgB_path)

H_A, W_A = imgA.shape[:2]
H_B, W_B = imgB.shape[:2]

# If your roma model has the high-level API:
with torch.no_grad():
    warp, certainty = roma.match(imgA_path, imgB_path, device=device)

# Sample sparse correspondences from dense warp
matches_norm, certainty_samples = roma.sample(warp, certainty)

# Convert normalized [-1,1] coords to pixel coordinates
kptsA, kptsB = roma.to_pixel_coordinates(matches_norm, H_A, W_A, H_B, W_B)

print("Num matches:", kptsA.shape[0])
print("First match:", kptsA[0], "->", kptsB[0])

: 